# Qwen3 Inference Server

Building an LLM serving stack from scratch — chat formatting, sampling,
KV caching, streaming, and a FastAPI wrapper.

**Runtime:** Runtime → Change runtime type → **L4 GPU** (needs ~10 GB VRAM).

## Install
Qwen3 requires transformers 4.51+. Older versions don't recognize the
architecture. `httpx` is used by the test client at the end.

In [8]:
!nvidia-smi
import torch; print(torch.cuda.get_device_name(0), torch.cuda.is_bf16_supported())

Wed Sep  9 19:19:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   34C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
!pip install -q "transformers>=4.51" accelerate fastapi uvicorn pydantic httpx

In [10]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())

NVIDIA L4
bf16 supported: True


## Load the model

Downloads ~8 GB on first run. A few minutes — not a hang.

- **bfloat16** — 2 bytes per number instead of 4. Matches the training dtype, so no quality loss. Falls back to fp16 on GPUs without bf16 support (anything pre-Ampere, e.g. T4).
- **device_map="auto"** — places weights on the GPU.
- **model.eval()** — disables training-only behavior like dropout.
- **STOP_IDS** — Qwen3 can end a turn on more than one token. Checking only `eos_token_id` lets generation run past the end and invent a fake user turn.

In [12]:
MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

# Qwen3 can stop on more than one token, so collect them all
STOP_IDS = set()
if tokenizer.eos_token_id is not None:
    STOP_IDS.add(tokenizer.eos_token_id)
cfg_eos = getattr(model.generation_config, "eos_token_id", None)
if isinstance(cfg_eos, list):
    STOP_IDS.update(cfg_eos)
elif cfg_eos is not None:
    STOP_IDS.add(cfg_eos)

print("device:", model.device, "| stop ids:", STOP_IDS)
print("VRAM used: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

device: cuda:0 | stop ids: {151643, 151645}
VRAM used: 8.0 GB


## TODO 1 — Prompt formatting

A language model has one skill: **continue text**. It has no concept of
"user" or "assistant." Chat is a *learned text layout* from post-training:

```
<|im_start|>user
What is a KV cache?<|im_end|>
<|im_start|>assistant
```

Reproduce that layout and you get chatbot behavior. Skip it and you get a
text-continuation machine that often answers a question with more questions.

- **add_generation_prompt=True** — appends the opening of the assistant's turn. Without it the model writes a *fake user message*. This is the #1 "why is my model broken" bug.
- **enable_thinking=False** — inserts a pre-closed empty `<think></think>` block, so the model skips reasoning and answers directly.
- **Whatever the template leaves open is what the model fills in.**

Returns `input_ids` (text as numbers) and `attention_mask` (which positions
are real). The mask looks pointless here — it matters enormously in TODO 4.

In [13]:
def prepare_inputs(prompt, system="You are a helpful assistant."):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    )
    return {k: v.to(model.device) for k, v in inputs.items()}


# Look at what the template actually produced
demo = tokenizer.apply_chat_template(
    [{"role": "system", "content": "You are a helpful assistant."},
     {"role": "user", "content": "What is a KV cache?"}],
    add_generation_prompt=True, enable_thinking=False, tokenize=False,
)
print(repr(demo))

inputs = prepare_inputs("What is a KV cache? Answer in two sentences.")
print(inputs["input_ids"].shape, inputs["attention_mask"].shape)

'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is a KV cache?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
torch.Size([1, 34]) torch.Size([1, 34])


## TODO 2 — Sampling

The model outputs **logits**: one raw score per vocab entry (~151k for Qwen3).
Not probabilities. Turning those into one chosen token happens entirely
*outside* the network — which is why APIs can expose these knobs without retraining.

**Temperature** divides the logits, changing the *gaps* between them.
Softmax exponentiates, so gap size drives everything.
Low T → wide gaps → predictable. High T → narrow gaps → varied.
T=0 would divide by zero, so it's special-cased to `argmax` (greedy, deterministic).

**Top-k** truncates the tail. Out of 151k tokens maybe 40 are plausible, but
150,960 tiny probabilities still add up to real mass. Find the k-th best score,
set everything below to `-inf`. Softmax maps `-inf` to exactly 0, and
renormalization is automatic.

**multinomial** rolls a weighted die. Returns shape `(batch, 1)` — already the
right shape to concatenate onto the sequence.

> Temperature and top-k **commute**. Temperature is monotonic, so it can't
> reorder anything, and top-k only depends on ranking.

In [14]:
def sample_next_token(logits, temperature=1.0, top_k=50):
    # logits shape: (batch, vocab_size) — scores for the next token
    if temperature == 0:
        return torch.argmax(logits, dim=-1, keepdim=True)

    logits = logits / temperature

    if top_k and top_k > 0:
        k = min(top_k, logits.size(-1))
        kth_value = torch.topk(logits, k, dim=-1).values[..., -1, None]
        logits = logits.masked_fill(logits < kth_value, float("-inf"))

    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)


# quick check
fake = torch.tensor([[1.0, 5.0, 2.0, 0.5]])
print("greedy  :", sample_next_token(fake, temperature=0).item())          # always 1
print("sampled :", [sample_next_token(fake, 1.0, 2).item() for _ in range(8)])  # only 1 or 2

greedy  : 1
sampled : [1, 1, 1, 1, 1, 1, 1, 1]


## TODO 3 — Generation without a cache (the slow baseline)

Generation is a loop: read everything so far, predict one token, append it,
repeat. That's all "generation" means.

This version is **deliberately wasteful** — it feeds the entire sequence back
through the model every single step. To produce token 100 it recomputes the
internals for tokens 1–99, which cannot possibly have changed.

- `logits[:, -1, :]` — the model predicts after *every* position (that's how training works). You only want the last one.
- The **mask grows in lockstep** with the sequence. Out of sync → silent garbage, no error.
- `[0, prompt_len:]` slices off the prompt. Returning it back to the caller is a real bug people ship.

In [15]:
@torch.inference_mode()
def generate_no_cache(inputs, max_new_tokens=64, temperature=0.0, top_k=50):
    generated_ids   = inputs["input_ids"]
    attention_mask  = inputs["attention_mask"]
    prompt_len      = generated_ids.shape[-1]

    for _ in range(max_new_tokens):
        outputs = model(
            input_ids=generated_ids,        # the ENTIRE sequence, every time
            attention_mask=attention_mask,
            use_cache=False,
        )
        next_logits = outputs.logits[:, -1, :]
        next_token  = sample_next_token(next_logits, temperature, top_k)

        if next_token.item() in STOP_IDS:
            break

        generated_ids  = torch.cat([generated_ids, next_token], dim=-1)
        attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=-1)

    completion_ids = generated_ids[0, prompt_len:]
    text = tokenizer.decode(completion_ids, skip_special_tokens=True)
    return text, completion_ids


text, ids = generate_no_cache(prepare_inputs("Say hello in one short sentence."), max_new_tokens=20)
print(text)

Hello!


In [17]:
CUDA = torch.cuda.is_available()

def sync():
    if CUDA:
        torch.cuda.synchronize()

print("sync defined | CUDA:", CUDA)

sync defined | CUDA: True


## TODO 4 — KV cache (the core of the project)

Each token produces a **query** ("what I'm looking for"), a **key**
("what I offer"), and a **value** ("my content"). Attention matches one
token's query against all visible keys, then blends their values.

The model is **causal** — a token only sees backwards. So token 5's key and
value are identical whether the sequence is 6 tokens or 600. They can never
change. **So store them instead of recomputing them.** That store is the KV cache.

This splits generation into two phases with completely different physics:

| | work | bottleneck | sets |
|---|---|---|---|
| **Prefill** | whole prompt, one parallel pass | **compute** | TTFT |
| **Decode** | one token per pass | **memory bandwidth** | tokens/sec |

Decode does almost no math, but must stream all 8 GB of weights from memory
*every token*. On an L4 (~300 GB/s): 300 ÷ 8 ≈ **37 tok/s ceiling**.

This distinction explains most of modern serving — batching and quantization
help decode because decode is bandwidth-bound.

> ⚠️ **The bug :** `input_ids` is length 1 but `attention_mask`
> stays **full length**. The mask describes which *positions* are visible, and
> the cached ones still are. A length-1 mask gives confident nonsense with no error.

In [18]:
@torch.inference_mode()
def generate_cached(inputs, max_new_tokens=64, temperature=0.0, top_k=50):
    input_ids      = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    # PREFILL — whole prompt, one pass
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=True)
    past_key_values = outputs.past_key_values
    next_logits     = outputs.logits[:, -1, :]

    sync()
    ttft_mark = time.perf_counter()

    completion_ids = []

    # DECODE — one token per pass
    for _ in range(max_new_tokens):
        next_token = sample_next_token(next_logits, temperature, top_k)
        tok = next_token.item()
        if tok in STOP_IDS:
            break
        completion_ids.append(tok)

        attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=-1)
        outputs = model(
            input_ids=next_token,            # ONE token
            attention_mask=attention_mask,   # FULL-LENGTH mask
            past_key_values=past_key_values,
            use_cache=True,
        )
        past_key_values = outputs.past_key_values
        next_logits     = outputs.logits[:, -1, :]

    text = tokenizer.decode(completion_ids, skip_special_tokens=True)
    return text, completion_ids, ttft_mark


t, ids, _ = generate_cached(prepare_inputs("Say hello in one short sentence."), 20)
print(t)

Hello!


## TODO 5 — Metrics

Three numbers that matter in serving:

- **TTFT** — time to first token. Driven by prefill, scales with prompt length. What a user perceives as "did it hang."
- **Tokens/sec** — driven by decode, roughly flat. Perceived typing speed.
- **Throughput** — total across all requests. What your GPU bill is made of.

Two traps that make numbers lie:
1. **No sync → you time kernel launches, not execution.** Results look impossibly fast.
2. **The first run is never representative** — CUDA compiles kernels on first use. Hence the warm-up call.

Use `time.perf_counter()`, not `time.time()`.

In [19]:
def compute_metrics(completion_ids, input_ids, latency_s, ttft_s=None):
    n_out = len(completion_ids)
    return {
        "input_tokens":  int(input_ids.shape[-1]),
        "output_tokens": n_out,
        "latency_s":     round(latency_s, 3),
        "ttft_s":        round(ttft_s, 3) if ttft_s is not None else None,
        "output_tokens_per_second": round(n_out / latency_s, 2) if latency_s > 0 else 0.0,
    }


def timed_run(fn, inputs, **kw):
    sync()
    t0 = time.perf_counter()
    result = fn(inputs, **kw)
    sync()
    latency = time.perf_counter() - t0

    if len(result) == 3:
        text, ids, ttft_mark = result
        ttft = ttft_mark - t0
    else:
        text, ids = result
        ttft = None
    return text, ids, latency, ttft


_ = timed_run(generate_cached, prepare_inputs("hi"), max_new_tokens=5)
print("warmed up")

warmed up


## Compare: cached vs uncached

The payoff cell. Watch two things:

**The speedup grows with `n`.** No-cache cost is quadratic in length; cached is
linear. Seeing that curve yourself beats reading about it ten times.

**Outputs must be identical at T=0.** The cache is pure memoization — same math,
same answer, less work. Not an approximation. If this prints `False` you have a
bug, and it's almost certainly the attention mask.

⏱ Slow by design — the no-cache runs at n=128 take a while. That *is* the lesson.
Drop to `[16, 32]` for a quick pass.

In [20]:
PROMPT = "Explain what a KV cache is in three sentences."

for n in [32, 64, 128]:
    _, ids_a, lat_a, _    = timed_run(generate_no_cache, prepare_inputs(PROMPT),
                                      max_new_tokens=n, temperature=0.0)
    txt, ids_b, lat_b, ttft = timed_run(generate_cached, prepare_inputs(PROMPT),
                                        max_new_tokens=n, temperature=0.0)
    print(f"n={n:3d} | no-cache {lat_a:6.2f}s ({len(ids_a)/lat_a:5.1f} tok/s)"
          f" | cached {lat_b:6.2f}s ({len(ids_b)/lat_b:5.1f} tok/s)"
          f" | {lat_a/lat_b:4.1f}x | ttft {ttft:.3f}s")

print("\noutput:", txt)
print("identical at T=0:", list(ids_a) == list(ids_b))

n= 32 | no-cache   1.63s ( 19.7 tok/s) | cached   1.60s ( 20.1 tok/s) |  1.0x | ttft 0.050s
n= 64 | no-cache   3.14s ( 20.4 tok/s) | cached   3.13s ( 20.5 tok/s) |  1.0x | ttft 0.051s
n=128 | no-cache   3.61s ( 20.2 tok/s) | cached   3.62s ( 20.4 tok/s) |  1.0x | ttft 0.051s

output: A KV cache, or Key-Value cache, is a type of memory storage system that stores frequently accessed data to speed up future access. It works by retaining recently or frequently used data in a fast memory, allowing for quicker retrieval compared to main memory or disk. KV caches are commonly used in databases, web applications, and other systems where performance and efficiency are critical.
identical at T=0: False


In [22]:
import sys

## TODO 6 — Streaming

Same generation, different delivery: hand back each new piece as it appears.

The tempting shortcut — decode each token alone and yield it — is **wrong**.
Tokens are chunks of *bytes*, not characters. An emoji or CJK character spans
several tokens, so decoding one alone yields a partial byte sequence.

The fix: decode the full list each step, send only the **string difference**.
Correct by construction — never splits a character, and joining all deltas
reproduces the text exactly. The `if delta` guard covers steps where a token
added bytes but hasn't completed a character.

`yield` makes this a generator, which is what FastAPI streams from.

In [23]:
@torch.inference_mode()
def generate_stream(inputs, max_new_tokens=64, temperature=0.0, top_k=50):
    input_ids      = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    outputs = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=True)
    past_key_values = outputs.past_key_values
    next_logits     = outputs.logits[:, -1, :]

    completion_ids, previous_text = [], ""

    for _ in range(max_new_tokens):
        next_token = sample_next_token(next_logits, temperature, top_k)
        tok = next_token.item()
        if tok in STOP_IDS:
            break
        completion_ids.append(tok)

        current_text = tokenizer.decode(completion_ids, skip_special_tokens=True)
        delta = current_text[len(previous_text):]
        if delta:
            yield delta
        previous_text = current_text

        attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=-1)
        outputs = model(
            input_ids=next_token,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            use_cache=True,
        )
        past_key_values = outputs.past_key_values
        next_logits     = outputs.logits[:, -1, :]


for chunk in generate_stream(prepare_inputs("Count from one to five."), 40):
    sys.stdout.write(chunk); sys.stdout.flush()

One, two, three, four, five.

## TODO 7 — Request and response schemas

Two forms: what people send you, what you send back. Pydantic validates
incoming JSON against these automatically and returns 422 for anything that
doesn't fit — **before any GPU work is scheduled**.

`Field(128, ge=1, le=512)` = default 128, must be between 1 and 512.
`Field(..., min_length=1)` — the `...` means required, no default.

**The limits are capacity control, not paperwork.** `max_new_tokens` decides
how long one request occupies your GPU. Unbounded, a single caller sends
`max_new_tokens: 1000000` and locks the machine for everyone. That's a DoS in
one line of JSON — and a buggy client does it by accident.

Metrics live in the *response body*, not just the logs, so any client can
measure your p99 without extra instrumentation.

In [24]:
from pydantic import BaseModel, Field
from typing import Optional

class GenerateRequest(BaseModel):
    prompt: str        = Field(..., min_length=1)
    system: str        = "You are a helpful assistant."
    max_new_tokens: int   = Field(128, ge=1, le=512)
    temperature: float    = Field(0.0, ge=0.0, le=2.0)
    top_k: int            = Field(50, ge=1, le=100)

class GenerateResponse(BaseModel):
    text: str
    ttft_s: Optional[float] = None
    latency_s: float
    input_tokens: int
    output_tokens: int
    output_tokens_per_second: float
    used_cache: bool

print("schemas ready")

schemas ready


## TODO 8 — Endpoints

An endpoint is a URL that runs a function. `@app.post(...)` wires them up.
GET = asking. POST = sending data.

- **`/health`** — load balancers ping this; if it stops answering they stop routing traffic here. Every real service has one.
- **`/generate`** — the whole notebook wired together: `prepare_inputs` → `timed_run(generate_cached)` → `compute_metrics`. The `req: GenerateRequest` type hint is what triggers validation.
- **`/stream`** — hands the generator to `StreamingResponse`, which flushes each chunk as it arrives.

**Note the asymmetry:** the streaming endpoint can't include metrics — by the
time latency is known, headers are long gone. Real APIs solve this with
server-sent events and a final `[DONE]` frame carrying usage stats.

Handlers are `def`, not `async def`. FastAPI runs sync handlers in a threadpool,
so a blocking GPU call doesn't stall the event loop.

⚠️ One model, one GPU — concurrent requests fight each other. Production puts a
queue and scheduler in front. That's where **continuous batching** comes in.

In [25]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

app = FastAPI(title="Qwen3 Inference Server")

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME}

@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest):
    inputs = prepare_inputs(req.prompt, req.system)
    text, ids, latency, ttft = timed_run(
        generate_cached, inputs,
        max_new_tokens=req.max_new_tokens,
        temperature=req.temperature,
        top_k=req.top_k,
    )
    metrics = compute_metrics(ids, inputs["input_ids"], latency, ttft)
    return GenerateResponse(text=text, used_cache=True, **metrics)

@app.post("/stream")
def stream(req: GenerateRequest):
    inputs = prepare_inputs(req.prompt, req.system)
    return StreamingResponse(
        generate_stream(inputs, req.max_new_tokens, req.temperature, req.top_k),
        media_type="text/plain",
    )

print("app ready")

app ready


## TODO 9 — Tests

`TestClient` calls the ASGI app **in-process** — no server, no port, no network.
That matters in Colab, which gives you no public port.

The last test is the one people skip: verify your guardrails actually **reject**.
A passing happy path proves nothing about what happens under abuse.

In [26]:
from fastapi.testclient import TestClient
client = TestClient(app)

r = client.get("/health")
assert r.status_code == 200 and r.json()["status"] == "ok"
print("health ok")

r = client.post("/generate", json={"prompt": "Say hello.", "max_new_tokens": 16})
assert r.status_code == 200
body = r.json()
assert body["output_tokens"] >= 1 and body["text"].strip()
print("generate ok:", body)

with client.stream("POST", "/stream",
                   json={"prompt": "Count to three.", "max_new_tokens": 16}) as r:
    assert r.status_code == 200
    chunks = list(r.iter_text())
assert "".join(chunks).strip()
print("stream ok:", "".join(chunks).strip())

r = client.post("/generate", json={"prompt": "hi", "max_new_tokens": 100000})
assert r.status_code == 422
print("guardrail ok — oversized request rejected")

health ok
generate ok: {'text': 'Hello! How can I assist you today? 😊', 'ttft_s': 0.054, 'latency_s': 0.594, 'input_tokens': 26, 'output_tokens': 11, 'output_tokens_per_second': 18.52, 'used_cache': True}
stream ok: One, two, three.
guardrail ok — oversized request rejected
